In [ ]:
DATA_SOURCES = ["archeology"]

# Setup

## Processor-Specific

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
from dotenv import load_dotenv
from torch.backends import cudnn

# enforce more deterministic behavior
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
cudnn.deterministic = True
cudnn.benchmark = False

sys.path.append("..")
load_dotenv("../../.env")

from processor.core.interaction_conductor.chat_interface import ChatInterface, ChatInterfaceOutputFormat
from processor.model.interface.impl.gpt import GPT
from processor.core.ir_system.ir_data_model import convert_multi_retriever_results_to_str

In [ ]:
llm_path = "model/weight/qwen3-8b"
embed_model_path = "model/weight/bge-base"
chat_interface = ChatInterface(llm_path, embed_model_path, "llm", DATA_SOURCES)
gpt = GPT("gpt-4o-mini")

## Benchmark-Specific

In [ ]:
import json
def read_jsonl(file_path):
    """
    Reads a JSON Lines (.jsonl) file and returns a list of Python dictionaries.
    
    Args:
        file_path (str): Path to the JSONL file.
    
    Returns:
        list: A list of dictionaries, one per line in the file.
    """
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:  # skip empty lines
                data.append(json.loads(line))
    return data
benchmark = read_jsonl(f"../../benchmark/benchmark_{DATA_SOURCES[0]}.jsonl")
def write_jsonl(filepath, data, append=False):
    """
    Write a list of JSON-serializable objects to a JSONL file.

    Args:
        filepath (str): Path to the output file.
        data (list): List of Python dictionaries or objects to write.
        append (bool): If True, append to existing file. Otherwise, overwrite.
    """
    mode = 'a' if append else 'w'
    with open(filepath, mode, encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

In [ ]:
def get_format_to_gpt(ci_output: ChatInterfaceOutputFormat):
    system_output = ci_output['system_response']
    state = ci_output['state']
    current_retrieval_results = ci_output['current_retrieval_results']

    return f"""SYSTEM OUTPUT:
```{system_output}```

STATE:
```{state}```

RETRIEVED DATA BY THE SYSTEM:
```{convert_multi_retriever_results_to_str(current_retrieval_results)}```
"""

In [ ]:
def get_initial_prompt_to_chatgpt(domain: str, question: str, initial_prompt: str):
    domain_expert_desc = f"a {domain} domain expert"
    if domain == "archeology":
        domain_expert_desc = "a domain expert in world cities, roman cities, radiocarbon data, world conflicts, and climate measurement exploration"
    return f"""You are simulating {domain_expert_desc}, who is interacting with a data assistant system to explore insights from an enterprise dataset. The system represents your information need as a set of target schemas, representing relevant table(s) for your question, along with a list of SQL statements, which if run sequentially on the (materialized) target schemas, will result in the answer to your question. You can criticize this representation if you think it does not represent your need correctly.

In this scenario, the system already has access to internal datasets. You (the simulated user) are already somewhat familiar with the topics of the datasets, as it is commonly used in your team or organization. You are not uploading a new dataset or asking about the existence of some datasets. Your task is to gradually explore or refine your information need about some aspect of the data. You do not begin with a precise question; rather, your curiosity evolves based on system responses and your domain expertise.

Here is a possible eventual goal (you do not know this yet, but may arrive at it through exploration):

{question}

Your behavior should reflect the following:
- You are familiar with the domain.
- You explore and refine your question step-by-step depending on the system's ability to surface relevant information.
- You may be vague or even explore tangents, just as a curious analyst would when exploring data without a clear goal.
- You will only arrive at the specific question above if the system's output correctly leads you there.
- The system may ask for clarification. Do not be too stubborn by not providing more details.

Continue your role as the domain expert. This is the conversation so far (again, provide response as if you are prompting the system directly):

YOU: {initial_prompt}"""

# INTERACTION

In [ ]:
from processor.model.llm_message import LLMMessage, Role
from processor.model.option import LLMOption

In [ ]:
index = 0
iteration = -1
INITIAL_PROMPT = benchmark[index]["interactive_initial_prompt"]
ITERATION_LIMIT = 15
print(f"Original (direct) question: {benchmark[index]["original_direct_question"]}")

In [ ]:
gpt_init_prompt = get_initial_prompt_to_chatgpt(
    DATA_SOURCES[0], benchmark[index]["original_direct_question"], INITIAL_PROMPT
)
gpt_messages = [LLMMessage(role=Role.SYSTEM.value, content=gpt_init_prompt)]
curr_user_prompt = INITIAL_PROMPT
print(f"=> CURRENT USER PROMPT: {curr_user_prompt}")

In [ ]:
iteration += 1
system_output = chat_interface.process_user_input(curr_user_prompt)
print(f"===> SYSTEM OUTPUT: {system_output}")
format_to_gpt = get_format_to_gpt(system_output)
if iteration == 0:
    gpt_messages[0]['content'] += f"\n{format_to_gpt}"
else:
    gpt_messages.append(LLMMessage(role=Role.USER.value, content=format_to_gpt))
updated_user_prompt = gpt.chat(gpt_messages, LLMOption(temperature=0))
gpt_messages.append(LLMMessage(role=Role.ASSISTANT.value, content=updated_user_prompt))
if updated_user_prompt.startswith("YOU:"):
    updated_user_prompt = updated_user_prompt[4:]
    updated_user_prompt = updated_user_prompt.strip()
curr_user_prompt = updated_user_prompt
print(f"=> CURRENT USER PROMPT: {curr_user_prompt}")

# FINAL

In [ ]:
# write_jsonl(f"benchmark_data/{DATA_SOURCES}_{index+1}.jsonl", gpt_messages, True)